In [2]:
import os
import numpy as np
from PIL import Image
import time
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
from fast_convolution import Convolution
from reshape import Reshape
from dense import Dense
from activations import ReLU, Softmax
from loss import categorical_cross_entropy, categorical_cross_entropy_prime, mse, mse_prime
from network import train, test
from fast_pooling import Pooling

In [8]:
# Cargar el dataset MNIST desde TensorFlow
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Tamaño fijo para redimensionar (1, 28, 28)
size = (28, 28)

# Función para codificar las etiquetas en formato one-hot
def one_hot_encode(labels, num_classes=10):
    one_hot_labels = np.zeros((len(labels), num_classes), dtype=int)
    for i, label in enumerate(labels):
        one_hot_labels[i, label] = 1
    return one_hot_labels

# Función para procesar las imágenes de MNIST y las etiquetas
def process_mnist_data(X, y):
    X_processed = []
    y_processed = []
    
    for i in range(len(X)):
        img = X[i]  # Imágenes en escala de grises ya están en formato (28, 28)
        
        # Redimensionar la imagen (aunque las imágenes de MNIST ya son 28x28)
        img = Image.fromarray(img).resize(size)

        # Convertir la imagen a array y normalizar
        img_array = np.array(img).astype('float32') / 255.0

        # Redimensionar para que sea (1, 28, 28)
        img_array = np.expand_dims(img_array, axis=0)

        # Agregar la imagen al dataset
        if img_array.shape == (1, 28, 28):
            X_processed.append(img_array)
        
            # Codificación one-hot para las etiquetas
            one_hot = one_hot_encode([y[i]])  # Codificamos la etiqueta de la imagen
            y_processed.append(one_hot[0])   # Agregamos el one-hot codificado
    
    return np.array(X_processed), np.array(y_processed)

# Procesar datos de entrenamiento y prueba
X_train, y_train = process_mnist_data(X_train, y_train)
X_test, y_test = process_mnist_data(X_test, y_test)

# Dividir los datos de entrenamiento en entrenamiento y validación (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Imprimir las formas de los datos
print("X_train shape:", X_train.shape)      # Debería ser (80% de num_imagenes, 1, 28, 28)
print("y_train shape:", y_train.shape)      # Debería ser (80% de num_imagenes, 10)
print("X_val shape:", X_val.shape)          # Debería ser (20% de num_imagenes, 1, 28, 28)
print("y_val shape:", y_val.shape)          # Debería ser (20% de num_imagenes, 10)
print("X_test shape:", X_test.shape)        # Debería ser (num_imagenes_test, 1, 28, 28)
print("y_test shape:", y_test.shape)        # Debería ser (num_imagenes_test, 10)

X_train shape: (48000, 1, 28, 28)
y_train shape: (48000, 10)
X_val shape: (12000, 1, 28, 28)
y_val shape: (12000, 10)
X_test shape: (10000, 1, 28, 28)
y_test shape: (10000, 10)


In [9]:
# Función para crear mini-batches a partir de los datos
def crear_mini_batches(X, y, batch_size, shuffle=True):
    # Mezclar los datos aleatoriamente
    if shuffle:
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]

    # Dividir los datos en batches del tamaño de batch_size
    mini_batches = []
    for i in range(0, X.shape[0], batch_size):
        X_mini_batch = X[i:i + batch_size]
        y_mini_batch = y[i:i + batch_size]
        mini_batches.append((X_mini_batch, y_mini_batch))

    return mini_batches

train_data = crear_mini_batches(X_train, y_train, batch_size=64)
val_data = crear_mini_batches(X_val, y_val, batch_size=64)
test_data = crear_mini_batches(X_test, y_test, batch_size=64)

In [11]:
# Definir la arquitectura de la red
net = [
    # Flatten: 16x13x13 -> 2704
    Reshape((1, 28, 28), 1*28*28),
    
    # Capa densa: 2704 -> 512
    Dense(1*28*28, 512),
    ReLU(),

    # Capa densa: 512 -> 128
    Dense(512, 128),
    ReLU(),
    
    # Capa densa: 128 -> 10
    Dense(128, 10),
    
    # Activación final
    Softmax()
]

# Entrenar la red
print("Iniciando entrenamiento...")
start_time = time.time()
epochs = 10
learning_rate = 0.1
history = train(train_data, val_data, net, categorical_cross_entropy, categorical_cross_entropy_prime, epochs, learning_rate)

# Evaluar el modelo
print("Evaluando el modelo...")
acc = test(X_test, y_test, net)
print(f"Precisión del modelo: {acc * 100:.2f}%")

# Tiempo total de ejecución
end_time = time.time()
print(f"Tiempo total de ejecución: {end_time - start_time:.2f} segundos")

Iniciando entrenamiento...
Epoch: 1/10 | Train Loss: 0.6713 | Train Acc: 0.8214 | Val Loss: 0.3298 | Val Acc: 0.9124
Tiempo total de ejecución: 5.13 segundos
Epoch: 2/10 | Train Loss: 0.3016 | Train Acc: 0.9201 | Val Loss: 0.2650 | Val Acc: 0.9308
Tiempo total de ejecución: 4.97 segundos
Epoch: 3/10 | Train Loss: 0.2476 | Train Acc: 0.9366 | Val Loss: 0.2289 | Val Acc: 0.9412
Tiempo total de ejecución: 4.91 segundos
Epoch: 4/10 | Train Loss: 0.2123 | Train Acc: 0.9473 | Val Loss: 0.2027 | Val Acc: 0.9481
Tiempo total de ejecución: 5.27 segundos
Epoch: 5/10 | Train Loss: 0.1860 | Train Acc: 0.9543 | Val Loss: 0.1829 | Val Acc: 0.9523
Tiempo total de ejecución: 5.03 segundos
Epoch: 6/10 | Train Loss: 0.1655 | Train Acc: 0.9596 | Val Loss: 0.1677 | Val Acc: 0.9557
Tiempo total de ejecución: 5.92 segundos
Epoch: 7/10 | Train Loss: 0.1487 | Train Acc: 0.9645 | Val Loss: 0.1555 | Val Acc: 0.9589
Tiempo total de ejecución: 5.97 segundos
Epoch: 8/10 | Train Loss: 0.1351 | Train Acc: 0.9690 | V

In [ ]:
# Rutas de las carpetas
train_path = 'Datos/Training'
test_path = 'Datos/Testing'
clases = ['glioma', 'meningioma', 'notumor', 'pituitary']
n = len(clases)

# Tamaño fijo para redimensionar (ancho, alto)
size = (64, 64)

# Función para cargar imágenes desde una carpeta con formato (3, 64, 64)
def load_images_from_folder(folder_path):
    X = []
    y = []
    for i, clase in enumerate(clases):
        clase_path = os.path.join(folder_path, clase)
        for img_name in os.listdir(clase_path):
            img_path = os.path.join(clase_path, img_name)
            with Image.open(img_path) as img:
                # Redimensiona la imagen al tamaño fijo
                img = img.resize(size)

                # Convertir a RGB si no lo es
                if img.mode != "RGB":
                    img = img.convert("RGB")

                # Convertir la imagen a array y normalizar
                img_array = np.array(img).astype('float32') / 255.0

                # Reorganizar la dimensión para que sea (3, 64, 64)
                img_array = np.transpose(img_array, (2, 0, 1))

                # Agregar la imagen al dataset
                if img_array.shape == (3, 64, 64):
                    X.append(img_array)

                    # Crear la codificación one-hot
                    one_hot = [0] * n
                    one_hot[i] = 1
                    y.append(one_hot)

    return np.array(X), np.array(y)

# Cargar datos de entrenamiento
X_train, y_train = load_images_from_folder(train_path)

# Dividir los datos de entrenamiento en entrenamiento y validación (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Cargar datos de prueba
X_test, y_test = load_images_from_folder(test_path)

# Imprimir las formas de los datos
print("X_train shape:", X_train.shape)      # Debería ser (80% de num_imagenes, 3, 64, 64)
print("y_train shape:", y_train.shape)      # Debería ser (80% de num_imagenes, n)
print("X_val shape:", X_val.shape)          # Debería ser (20% de num_imagenes, 3, 64, 64)
print("y_val shape:", y_val.shape)          # Debería ser (20% de num_imagenes, n)
print("X_test shape:", X_test.shape)        # Debería ser (num_imagenes_test, 3, 64, 64)
print("y_test shape:", y_test.shape)        # Debería ser (num_imagenes_test, n)

X_train shape: (4569, 3, 64, 64)
y_train shape: (4569, 4)
X_val shape: (1143, 3, 64, 64)
y_val shape: (1143, 4)
X_test shape: (1311, 3, 64, 64)
y_test shape: (1311, 4)


In [13]:
train_data = crear_mini_batches(X_train, y_train, batch_size=64)
val_data = crear_mini_batches(X_val, y_val, batch_size=64)
test_data = crear_mini_batches(X_test, y_test, batch_size=64)

In [ ]:
# Definir la arquitectura de la red
net = [
    # Capa 1: Input 1x28x28 -> Conv 3x3 -> 16x26x26
    Convolution((1, 28, 28), 3, 16),       # 16 filtros de 3x3
    ReLU(),
    
    # Pooling: 16x26x26 -> 16x13x13
    Pooling(2, 2),                         # MaxPooling 2x2 (stride=2)
    
    # Flatten: 16x13x13 -> 2704
    Reshape((16, 13, 13), 16*13*13),
    
    # Capa densa: 2704 -> 128
    Dense(16*13*13, 128),
    ReLU(),
    
    # Capa densa: 128 -> 10
    Dense(128, 10),
    
    # Activación final
    Softmax()
]

# Entrenar la red
print("Iniciando entrenamiento...")
start_time = time.time()
epochs = 1
learning_rate = 0.1
history = train(train_data, val_data, net, categorical_cross_entropy, categorical_cross_entropy_prime, epochs, learning_rate)


# Evaluar el modelo
print("Evaluando el modelo...")
acc = test(X_test, y_test, net)
print(f"Precisión del modelo: {acc * 100:.2f}%")

# Tiempo total de ejecución
end_time = time.time()
print(f"Tiempo total de ejecución: {end_time - start_time:.2f} segundos")

Iniciando entrenamiento...
Epoch: 1/1 | Train Loss: 1.2528 | Train Acc: 0.4771 | Val Loss: 1.0367 | Val Acc: 0.5293
Tiempo total de ejecución: 735.11 segundos
Evaluando el modelo...
Precisión del modelo: 50.34%
Tiempo total de ejecución: 892.99 segundos
